# PINN a posteriori estimator on the unit square

This notebook trains a PINN for the manufactured Poisson problem on $\Omega=[0,1]^2$, and computes certified interval enclosures for:

- the interior residual norm $\|r_\theta\|_{L^2(\Omega)}$ using **interval Hessian bounds**,
- the boundary mismatch norm $\|u_\theta-g\|_{L^2(\partial\Omega)}$,
- and the combined estimator $\eta_\theta$.

## 1) Setup and imports

In [ ]:
from __future__ import annotations

import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

repo_root = Path.cwd()
while not (repo_root / "src" / "intervalnets").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root / "src") not in __import__('sys').path:
    __import__('sys').path.insert(0, str(repo_root / "src"))

from intervalnets import Interval, IntervalTensor, enable_interval_eval

enable_interval_eval(enclosure_mode="slope")

dtype = torch.float64
device = torch.device("cpu")
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"torch={torch.__version__}, dtype={dtype}, seed={SEED}")

## 2) Problem definition (PDE, exact solution, forcing, BC)

In [ ]:
PI = math.pi

def u_exact(xy: torch.Tensor) -> torch.Tensor:
    x = xy[:, 0:1]
    y = xy[:, 1:2]
    return torch.sin(PI * x) * torch.sin(PI * y)


def forcing_f(xy: torch.Tensor) -> torch.Tensor:
    x = xy[:, 0:1]
    y = xy[:, 1:2]
    return 2.0 * (PI ** 2) * torch.sin(PI * x) * torch.sin(PI * y)


def g_boundary(xy: torch.Tensor) -> torch.Tensor:
    return torch.zeros((xy.shape[0], 1), dtype=xy.dtype, device=xy.device)

## 3) PINN model

In [ ]:
def make_pinn(width: int = 64, hidden_layers: int = 4) -> nn.Sequential:
    layers: list[nn.Module] = [nn.Linear(2, width), nn.Tanh()]
    for _ in range(hidden_layers - 1):
        layers += [nn.Linear(width, width), nn.Tanh()]
    layers += [nn.Linear(width, 1)]
    return nn.Sequential(*layers)

model = make_pinn(width=64, hidden_layers=4).to(device=device, dtype=dtype)
model

## 4) Training data sampling (interior + boundary)

In [ ]:
def sample_interior(n: int) -> torch.Tensor:
    return torch.rand((n, 2), dtype=dtype, device=device)


def sample_boundary(n_per_edge: int) -> torch.Tensor:
    t = torch.rand((n_per_edge, 1), dtype=dtype, device=device)
    z = torch.zeros_like(t)
    o = torch.ones_like(t)
    return torch.cat([
        torch.cat([t, z], dim=1),
        torch.cat([t, o], dim=1),
        torch.cat([z, t], dim=1),
        torch.cat([o, t], dim=1),
    ], dim=0)


def laplacian_u_autograd(model: nn.Module, xy: torch.Tensor) -> torch.Tensor:
    xy_req = xy.detach().clone().requires_grad_(True)
    u = model(xy_req)
    grad_u = torch.autograd.grad(u, xy_req, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(grad_u[:, 0:1], xy_req, grad_outputs=torch.ones_like(grad_u[:, 0:1]), create_graph=True)[0][:, 0:1]
    u_yy = torch.autograd.grad(grad_u[:, 1:2], xy_req, grad_outputs=torch.ones_like(grad_u[:, 1:2]), create_graph=True)[0][:, 1:2]
    return u_xx + u_yy


def residual_r(model: nn.Module, xy: torch.Tensor) -> torch.Tensor:
    return -laplacian_u_autograd(model, xy) - forcing_f(xy)

## 5) Training loop

In [ ]:
EPOCHS = 1200
LR = 1e-3
W_INTERIOR = 1.0
W_BOUNDARY = 20.0
N_INTERIOR = 1024
N_BDRY_PER_EDGE = 256

opt = torch.optim.Adam(model.parameters(), lr=LR)
history = {"total": [], "interior": [], "boundary": []}

for epoch in range(1, EPOCHS + 1):
    xi = sample_interior(N_INTERIOR)
    xb = sample_boundary(N_BDRY_PER_EDGE)

    ri = residual_r(model, xi)
    bm = model(xb) - g_boundary(xb)

    li = torch.mean(ri ** 2)
    lb = torch.mean(bm ** 2)
    loss = W_INTERIOR * li + W_BOUNDARY * lb

    opt.zero_grad()
    loss.backward()
    opt.step()

    history["total"].append(float(loss.detach().cpu()))
    history["interior"].append(float(li.detach().cpu()))
    history["boundary"].append(float(lb.detach().cpu()))

    if epoch % 200 == 0:
        print(f"epoch={epoch:4d} total={history['total'][-1]:.3e} interior={history['interior'][-1]:.3e} boundary={history['boundary'][-1]:.3e}")

model.eval();

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.semilogy(history["total"], label="total")
ax.semilogy(history["interior"], label="interior")
ax.semilogy(history["boundary"], label="boundary")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_title("PINN training curves")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 6) Sanity diagnostics (sampled residual and boundary mismatch)

In [ ]:
@torch.no_grad()
def sample_l2_norm(values: torch.Tensor) -> float:
    return float(torch.sqrt(torch.mean(values**2)).cpu())

xi_diag = sample_interior(20000)
xb_diag = sample_boundary(2000)

r_diag = residual_r(model, xi_diag).detach()
b_diag = (model(xb_diag) - g_boundary(xb_diag)).detach()
u_err_diag = (model(xi_diag) - u_exact(xi_diag)).detach()

empirical = {
    "||r_theta||_{L2(Ω), MC}": sample_l2_norm(r_diag),
    "||u_theta-g||_{L2(∂Ω), MC}": sample_l2_norm(b_diag),
    "||u_theta-u*||_{L2(Ω), MC}": sample_l2_norm(u_err_diag),
}
for k, v in empirical.items():
    print(f"{k:35s}: {v:.6e}")

## 7) Certified interval enclosure of residual $L^2(\Omega)$ (direct Hessian-based)

In [ ]:
def interval_abs(iv: Interval) -> Interval:
    lo = float(iv.lower)
    hi = float(iv.upper)
    if lo >= 0.0:
        return Interval.from_bounds(lo, hi)
    if hi <= 0.0:
        return Interval.from_bounds(-hi, -lo)
    return Interval.from_bounds(0.0, max(-lo, hi))


def sin_interval(a: float, b: float) -> Interval:
    points = [a, b]
    k_start = math.ceil((a - math.pi / 2.0) / math.pi)
    k_end = math.floor((b - math.pi / 2.0) / math.pi)
    for k in range(k_start, k_end + 1):
        points.append(math.pi / 2.0 + k * math.pi)
    vals = [math.sin(t) for t in points]
    return Interval.from_bounds(min(vals), max(vals))


def forcing_interval_on_box(box: IntervalTensor) -> Interval:
    x_lo, y_lo = float(box.lower[0]), float(box.lower[1])
    x_hi, y_hi = float(box.upper[0]), float(box.upper[1])
    sx = sin_interval(math.pi * x_lo, math.pi * x_hi)
    sy = sin_interval(math.pi * y_lo, math.pi * y_hi)
    return Interval.point(2.0 * (math.pi ** 2)) * sx * sy


def residual_pointwise_power_bounds(model: nn.Module, box: IntervalTensor) -> Interval:
    hess = model.eval_hessian(box)
    u_xx = Interval(hess.lower[0][0][0], hess.upper[0][0][0])
    u_yy = Interval(hess.lower[0][1][1], hess.upper[0][1][1])
    lap = u_xx + u_yy
    f_iv = forcing_interval_on_box(box)
    r_iv = (Interval.point(0.0) - lap) - f_iv
    abs_r = interval_abs(r_iv)
    return abs_r * abs_r


def split_box(box: IntervalTensor):
    widths = [float(box.upper[i] - box.lower[i]) for i in range(len(box.lower))]
    dim = int(np.argmax(widths))
    mid = 0.5 * (float(box.lower[dim]) + float(box.upper[dim]))
    lo1 = list(box.lower)
    up1 = list(box.upper)
    lo2 = list(box.lower)
    up2 = list(box.upper)
    up1[dim] = mid
    lo2[dim] = mid
    return IntervalTensor.from_bounds(lo1, up1), IntervalTensor.from_bounds(lo2, up2)


def box_volume(box: IntervalTensor) -> float:
    vol = 1.0
    for lo, hi in zip(box.lower, box.upper):
        vol *= float(hi - lo)
    return vol


def certified_residual_l2(model: nn.Module, domain: IntervalTensor, iterations: int = 6, theta: float = 0.6, return_boxes: bool = False):
    boxes = [domain]
    last_indicators = []
    for _ in range(iterations):
        indicators = []
        for box in boxes:
            iv = residual_pointwise_power_bounds(model, box)
            indicators.append((float(iv.upper) - float(iv.lower)) * box_volume(box))
        last_indicators = indicators
        total = sum(indicators)
        order = np.argsort(indicators)[::-1]
        marked = []
        running = 0.0
        target = theta * total
        for idx in order:
            marked.append(int(idx))
            running += indicators[int(idx)]
            if running >= target:
                break
        marked_set = set(marked)
        new_boxes = []
        for i, box in enumerate(boxes):
            if i in marked_set:
                a, b = split_box(box)
                new_boxes.extend([a, b])
            else:
                new_boxes.append(box)
        boxes = new_boxes

    integral = Interval.point(0.0)
    for box in boxes:
        power_iv = residual_pointwise_power_bounds(model, box)
        weighted = Interval.from_bounds(float(power_iv.lower) * box_volume(box), float(power_iv.upper) * box_volume(box))
        integral = integral + weighted
    result = Interval.from_bounds(math.sqrt(max(0.0, float(integral.lower))), math.sqrt(max(0.0, float(integral.upper))))
    if return_boxes:
        if not last_indicators:
            last_indicators = [((float(residual_pointwise_power_bounds(model, b).upper) - float(residual_pointwise_power_bounds(model, b).lower)) * box_volume(b)) for b in boxes]
        return result, boxes, last_indicators
    return result

DOMAIN_2D = IntervalTensor.from_bounds([0.0, 0.0], [1.0, 1.0])
RES_ITERS = 6
RES_THETA = 0.6
residual_l2_iv, residual_boxes, residual_indicators = certified_residual_l2(model, DOMAIN_2D, iterations=RES_ITERS, theta=RES_THETA, return_boxes=True)
print(f"Certified ||r_theta||_L2(Ω) ∈ [{float(residual_l2_iv.lower):.6e}, {float(residual_l2_iv.upper):.6e}], width={float(residual_l2_iv.upper-residual_l2_iv.lower):.3e}")

## 8) Certified interval enclosure of boundary trace $L^2(\partial\Omega)$

In [ ]:
def make_edge_map(kind: str) -> nn.Linear:
    layer = nn.Linear(1, 2, bias=True).to(device=device, dtype=dtype)
    with torch.no_grad():
        if kind == "t0":
            layer.weight[:] = torch.tensor([[1.0], [0.0]], dtype=dtype)
            layer.bias[:] = torch.tensor([0.0, 0.0], dtype=dtype)
        elif kind == "t1":
            layer.weight[:] = torch.tensor([[1.0], [0.0]], dtype=dtype)
            layer.bias[:] = torch.tensor([0.0, 1.0], dtype=dtype)
        elif kind == "0t":
            layer.weight[:] = torch.tensor([[0.0], [1.0]], dtype=dtype)
            layer.bias[:] = torch.tensor([0.0, 0.0], dtype=dtype)
        elif kind == "1t":
            layer.weight[:] = torch.tensor([[0.0], [1.0]], dtype=dtype)
            layer.bias[:] = torch.tensor([1.0, 0.0], dtype=dtype)
        else:
            raise ValueError(kind)
    for p in layer.parameters():
        p.requires_grad_(False)
    return layer

edge_models = {
    "edge1:(t,0)": nn.Sequential(make_edge_map("t0"), model),
    "edge2:(t,1)": nn.Sequential(make_edge_map("t1"), model),
    "edge3:(0,t)": nn.Sequential(make_edge_map("0t"), model),
    "edge4:(1,t)": nn.Sequential(make_edge_map("1t"), model),
}

DOMAIN_1D = IntervalTensor.from_bounds([0.0], [1.0])
BND_ITERS = 5
BND_THETA = 0.6
BND_FORWARD_SPLITS = 3

edge_norm_intervals = {}
for name, edge_model in edge_models.items():
    iv = edge_model.lpnorm(DOMAIN_1D, p=2.0, iterations=BND_ITERS, theta=BND_THETA, forward_refine_splits=BND_FORWARD_SPLITS)
    edge_norm_intervals[name] = iv
    print(f"{name:14s}: [{float(iv.lower):.6e}, {float(iv.upper):.6e}] width={float(iv.upper-iv.lower):.3e}")

sum_lower = 0.0
sum_upper = 0.0
for iv in edge_norm_intervals.values():
    lk = max(0.0, float(iv.lower))
    uk = max(0.0, float(iv.upper))
    sum_lower += lk * lk
    sum_upper += uk * uk
boundary_l2_iv = Interval.from_bounds(math.sqrt(sum_lower), math.sqrt(sum_upper))
print(f"Global certified ||u_theta-g||_L2(∂Ω) ∈ [{float(boundary_l2_iv.lower):.6e}, {float(boundary_l2_iv.upper):.6e}]")

## 9) Combined a posteriori estimator

In [ ]:
eta_iv = residual_l2_iv + boundary_l2_iv
print(f"η interval: [{float(eta_iv.lower):.6e}, {float(eta_iv.upper):.6e}], width={float(eta_iv.upper-eta_iv.lower):.3e}")

# Optional: demonstrate new Sobolev order argument (W^{2,2}).
w12_iv = model.sobolev_norm(DOMAIN_2D, p=2.0, order=1, iterations=3)
w22_iv = model.sobolev_norm(DOMAIN_2D, p=2.0, order=2, iterations=3)
print(f"W12 interval: [{float(w12_iv.lower):.6e}, {float(w12_iv.upper):.6e}]")
print(f"W22 interval: [{float(w22_iv.lower):.6e}, {float(w22_iv.upper):.6e}]")

## 10) PINN field visualizations

We visualize:
1. PINN solution $u_\theta$,
2. absolute error $|u_\theta-u^*|$,
3. local certified residual indicators used in adaptive refinement.

In [ ]:
# (a) solution and (b) absolute error on a regular grid
N_PLOT = 121
x = torch.linspace(0.0, 1.0, N_PLOT, dtype=dtype, device=device)
y = torch.linspace(0.0, 1.0, N_PLOT, dtype=dtype, device=device)
X, Y = torch.meshgrid(x, y, indexing="ij")
XY = torch.stack([X.reshape(-1), Y.reshape(-1)], dim=1)

with torch.no_grad():
    U_pred = model(XY).reshape(N_PLOT, N_PLOT).cpu().numpy()
    U_true = u_exact(XY).reshape(N_PLOT, N_PLOT).cpu().numpy()
ABS_ERR = np.abs(U_pred - U_true)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)

im0 = axes[0].imshow(U_pred.T, origin="lower", extent=[0, 1, 0, 1], cmap="viridis", aspect="equal")
axes[0].set_title("(a) PINN solution $u_\theta$")
axes[0].set_xlabel("x"); axes[0].set_ylabel("y")
fig.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(ABS_ERR.T, origin="lower", extent=[0, 1, 0, 1], cmap="magma", aspect="equal")
axes[1].set_title("(b) Absolute error $|u_\theta-u^*|$")
axes[1].set_xlabel("x"); axes[1].set_ylabel("y")
fig.colorbar(im1, ax=axes[1], fraction=0.046)

# (c) local certified residual indicators per adaptive cell
axes[2].set_title("(c) Local residual indicators")
axes[2].set_xlim(0, 1); axes[2].set_ylim(0, 1)
axes[2].set_aspect("equal")
axes[2].set_xlabel("x"); axes[2].set_ylabel("y")

ind = np.array(residual_indicators, dtype=float)
if ind.size == 0:
    ind = np.array([0.0])
ind_min = float(ind.min())
ind_max = float(ind.max())

for box, val in zip(residual_boxes, residual_indicators):
    x0, y0 = float(box.lower[0]), float(box.lower[1])
    x1, y1 = float(box.upper[0]), float(box.upper[1])
    if ind_max > ind_min:
        alpha = 0.15 + 0.85 * ((float(val) - ind_min) / (ind_max - ind_min))
    else:
        alpha = 0.4
    rect = plt.Rectangle((x0, y0), x1 - x0, y1 - y0, facecolor=(0.1, 0.2, 0.8, alpha), edgecolor="black", linewidth=0.3)
    axes[2].add_patch(rect)

sm = plt.cm.ScalarMappable(cmap=plt.cm.Blues, norm=plt.Normalize(vmin=ind_min, vmax=ind_max if ind_max > ind_min else ind_min + 1.0))
sm.set_array([])
fig.colorbar(sm, ax=axes[2], fraction=0.046, label="indicator magnitude")

plt.show()

print(f"Grid max abs error: {ABS_ERR.max():.6e}")
print(f"Adaptive cells in indicator map: {len(residual_boxes)}")

## 11) Summary table and conclusions

In [ ]:
rows = [
    {
        "metric": "Residual ||r_theta||_L2(Ω)",
        "empirical": empirical["||r_theta||_{L2(Ω), MC}"],
        "cert_lower": float(residual_l2_iv.lower),
        "cert_upper": float(residual_l2_iv.upper),
        "width": float(residual_l2_iv.upper - residual_l2_iv.lower),
    },
    {
        "metric": "Boundary ||u_theta-g||_L2(∂Ω)",
        "empirical": empirical["||u_theta-g||_{L2(∂Ω), MC}"],
        "cert_lower": float(boundary_l2_iv.lower),
        "cert_upper": float(boundary_l2_iv.upper),
        "width": float(boundary_l2_iv.upper - boundary_l2_iv.lower),
    },
    {
        "metric": "Combined η",
        "empirical": np.nan,
        "cert_lower": float(eta_iv.lower),
        "cert_upper": float(eta_iv.upper),
        "width": float(eta_iv.upper - eta_iv.lower),
    },
]
for name, iv in edge_norm_intervals.items():
    rows.append({"metric": f"Per-edge {name}", "empirical": np.nan, "cert_lower": float(iv.lower), "cert_upper": float(iv.upper), "width": float(iv.upper-iv.lower)})

pd.DataFrame(rows)

### Interpretation

- The residual enclosure is now computed directly from certified Hessian bounds and interval forcing bounds on each adaptive cell.
- The boundary term remains certified via edge-wise `lpnorm` computations.
- The practical estimator $\eta_\theta = \|r_\theta\|_{L^2(\Omega)} + \|u_\theta-g\|_{L^2(\partial\Omega)}$ is therefore fully interval-certified in this notebook.